# **Import Library**

In [ ]:
import geopandas as gpd
import pandas as pd
from scipy import sparse
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import Queen, WSP, W
from numpy import sqrt
import statsmodels.api as sm
!pip install esda splot libpysal geopandas
from esda.moran import Moran
from statsmodels.stats.outliers_influence import variance_inflation_factor
from operator import itemgetter
from scipy.stats import kstest, norm

# **Load Data**

In [ ]:
gdf = gpd.read_file("Jawa Tengah.zip")

In [ ]:
spasial_df = pd.read_excel("Spasial_Jawa_Tengah_Cleaned.xlsx")
spasial_df = spasial_df.drop(columns = ["Latitude","Longitude"])

# **Statistika Deskriptif**

In [ ]:
gdf.head()

,NAMOBJ,FCODE,REMARK,METADATA,SRS_ID,KDBBPS,KDCBPS,KDCPUM,KDEBPS,KDEPUM,...,WADMKD,WADMKK,WADMPR,WIADKC,WIADKK,WIADPR,WIADKD,SHAPE_Leng,SHAPE_Area,geometry
0,Banjarnegara,BA03050040,None,TASWIL5000020230907KABKOTA,4326,None,None,None,None,None,...,None,Banjarnegara,Jawa Tengah,None,None,None,0,2.297379,0.093763,"POLYGON Z ((109.91691 -7.18985 0, 109.91746 -7..."
1,Banyumas,BA03050040,None,TASWIL5000020230907KABKOTA,4326,None,None,None,None,None,...,None,Banyumas,Jawa Tengah,None,None,None,0,2.197132,0.113957,"POLYGON Z ((109.3617 -7.48538 0, 109.36152 -7...."
2,Batang,BA03050040,None,TASWIL5000020230907KABKOTA,4326,None,None,None,None,None,...,None,Batang,Jawa Tengah,None,None,None,0,1.531137,0.070157,"POLYGON Z ((109.71086 -6.86701 0, 109.71113 -6..."
3,Blora,BA03050040,None,TASWIL5000020230907KABKOTA,4326,None,None,None,None,None,...,None,Blora,Jawa Tengah,None,None,None,0,2.588665,0.160200,"POLYGON Z ((111.26338 -6.85576 0, 111.26338 -6..."
4,Boyolali,BA03050040,None,TASWIL5000020230907KABKOTA,4326,None,None,None,None,None,...,None,Boyolali,Jawa Tengah,None,None,None,0,3.034328,0.089820,"POLYGON Z ((110.83881 -7.26261 0, 110.83741 -7..."


In [ ]:
gdf.describe()

,LUASWH,TIPADM,WIADKD,SHAPE_Leng,SHAPE_Area
count,35.000000,35.000000,35.0,35.000000,35.000000
mean,981.071558,4.171429,0.0,1.953899,0.080331
std,582.048388,0.382385,0.0,0.951163,0.047666
min,18.558259,4.000000,0.0,0.270626,0.001520
25%,752.266064,4.000000,0.0,1.562397,0.061649
50%,1008.124240,4.000000,0.0,1.916551,0.082506
75%,1141.154380,4.000000,0.0,2.541063,0.093425
max,2323.926116,5.000000,0.0,4.838730,0.190380


In [ ]:
spasial_df.head()

,Kab/Kota,Y,RLS,AHH,TPT,APD
0,Banjarnegara,14.71,6.910,74.875,5.57,2352.02
1,Banyumas,11.95,7.925,74.440,6.18,3852.98
2,Batang,8.73,7.175,75.295,5.67,1943.91
3,Blora,11.42,7.330,75.145,3.67,2624.07
4,Boyolali,9.63,8.215,76.650,3.16,2354.35


In [ ]:
spasial_df.describe()

,Y,RLS,AHH,TPT,APD
count,35.000000,35.000000,35.000000,35.000000,35.000000
mean,10.128571,8.359143,75.644429,4.519714,2476.441429
std,3.211398,1.383067,1.828924,1.494546,814.954276
min,4.030000,6.420000,70.340000,2.350000,983.640000
25%,7.555000,7.397500,74.670000,3.500000,2118.945000
50%,9.630000,7.925000,75.470000,3.970000,2403.010000
75%,11.995000,9.250000,76.830000,5.320000,2782.595000
max,15.710000,11.550000,78.375000,8.350000,5231.590000


# **WEIGHT MATRIX**

In [ ]:
gdf["centroid"] = gdf.geometry.centroid
gdf["centroid_lon"] = gdf.centroid.x
gdf["centroid_lat"] = gdf.centroid.y
print(gdf[["centroid", "centroid_lon", "centroid_lat"]])

                      centroid  centroid_lon  centroid_lat
0   POINT (109.65705 -7.35133)    109.657052     -7.351327
1   POINT (109.17579 -7.45551)    109.175788     -7.455509
2     POINT (109.86159 -7.021)    109.861593     -7.021002
3   POINT (111.38744 -7.07444)    111.387435     -7.074438
4    POINT (110.6525 -7.41652)    110.652499     -7.416522
5   POINT (108.92953 -7.06277)    108.929526     -7.062772
6   POINT (108.88984 -7.48893)    108.889839     -7.488927
7     POINT (110.634 -6.91077)    110.634005     -6.910774
8   POINT (110.92718 -7.11764)    110.927178     -7.117637
9     POINT (110.76813 -6.549)    110.768129     -6.549003
10  POINT (111.01924 -7.61414)    111.019242     -7.614139
11  POINT (109.61735 -7.65496)    109.617353     -7.654964
12  POINT (110.15734 -7.03869)    110.157338     -7.038695
13  POINT (110.61945 -7.68582)    110.619452     -7.685818
14  POINT (110.22011 -7.47655)    110.220111     -7.476555
15  POINT (109.67716 -6.89341)    109.677155     -6.8934

/tmp/ipython-input-1311769737.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid
/tmp/ipython-input-1311769737.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lon"] = gdf.centroid.x
/tmp/ipython-input-1311769737.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lat"] = gdf.centroid.y


## Memilih kolom yang akan dipakai pada gdf

In [ ]:
df = pd.DataFrame(gdf[["NAMOBJ","centroid", "centroid_lon", "centroid_lat"]])

In [ ]:
df["Index"] = df.index

In [ ]:
df = df[["Index",'NAMOBJ',"centroid", "centroid_lon", "centroid_lat"]]

In [ ]:
df

,Index,NAMOBJ,centroid,centroid_lon,centroid_lat
0,0,Banjarnegara,POINT (109.65705 -7.35133),109.657052,-7.351327
1,1,Banyumas,POINT (109.17579 -7.45551),109.175788,-7.455509
2,2,Batang,POINT (109.86159 -7.021),109.861593,-7.021002
3,3,Blora,POINT (111.38744 -7.07444),111.387435,-7.074438
4,4,Boyolali,POINT (110.6525 -7.41652),110.652499,-7.416522
5,5,Brebes,POINT (108.92953 -7.06277),108.929526,-7.062772
6,6,Cilacap,POINT (108.88984 -7.48893),108.889839,-7.488927
7,7,Demak,POINT (110.634 -6.91077),110.634005,-6.910774
8,8,Grobogan,POINT (110.92718 -7.11764),110.927178,-7.117637
9,9,Jepara,POINT (110.76813 -6.549),110.768129,-6.549003


## Menghitung jarak dengan jarak Haversine

Karena ini adalah data spasial yang cukup besar, lebih akurat apabila menggunakan jarak Haversine

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    R = 6371  # Jari-jari bumi dalam km
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

Distance Matrix tanpa standardisasi dan neighbor

In [ ]:
coords = df[['centroid_lon', "centroid_lat"]].to_numpy()
n = len(coords)
dist = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist[i, j] = haversine(coords[i,0], coords[i,1], coords[j,0], coords[j,1])

In [ ]:
w = Queen.from_dataframe(gdf)

/tmp/ipython-input-38079147.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)


In [ ]:
yesyes = np.zeros((35,35))
yes = w.neighbors
for i in yes:
  for j in yes[i]:
    yesyes[i][j] = dist[i][j]

## Standardisasi Baris Bobot Matriks

In [ ]:
def standard(array):
  for i,j in enumerate(array):
    array[i] = j/sum(j)
  return array

In [ ]:
final = standard(yesyes)

In [ ]:
final

array([[0.        , 0.24610604, 0.19531803, ..., 0.        , 0.        ,
        0.12927293],
       [0.16953397, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.25096599, 0.        , 0.        , ..., 0.        , 0.        ,
        0.25758005],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.25193321],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.10715765, 0.        , 0.16617102, ..., 0.11567232, 0.        ,
        0.        ]])

In [ ]:
for i in final:
  print(sum(i))

1.0
0.9999999999999998
1.0
1.0
0.9999999999999997
1.0
1.0
1.0
1.0000000000000002
0.9999999999999999
1.0
1.0
0.9999999999999999
1.0
1.0
1.0
1.0
1.0
1.0
0.9999999999999999
1.0
0.9999999999999999
1.0
1.0
1.0
1.0
1.0000000000000002
1.0
0.9999999999999997
1.0
1.0
1.0
0.9999999999999999
1.0
1.0000000000000002


# **Uji Moran**

$I = \frac{n}{W} \cdot \frac{\sum_i \sum_j w_{ij} (x_i - \bar{x})(x_j - \bar{x})}{\sum_i (x_i - \bar{x})^2}$

In [ ]:
W_array = final.copy().astype(float)
y = spasial_df['Y'].values

neighbors = {i: np.where(W_array[i] > 0)[0].tolist() for i in range(W_array.shape[0])}
weights = {i: W_array[i, W_array[i] > 0].tolist() for i in range(W_array.shape[0])}

w = W(neighbors, weights)
w.transform = "R"

moran = Moran(y, w, permutations=999)

print("Moran’s I:", moran.I)
print("Expected I:", moran.EI)
print("Z-score:", moran.z_sim)
print("P-value:", moran.p_sim)

Moran’s I: 0.15535806091745252
Expected I: -0.029411764705882353
Z-score: 1.546741541284157
P-value: 0.07


# **SPATIAL LAG X MODEL**

In [ ]:
spasial_df

,Kab/Kota,Y,RLS,AHH,TPT,APD
0,Banjarnegara,14.71,6.910,74.875,5.57,2352.02
1,Banyumas,11.95,7.925,74.440,6.18,3852.98
2,Batang,8.73,7.175,75.295,5.67,1943.91
3,Blora,11.42,7.330,75.145,3.67,2624.07
4,Boyolali,9.63,8.215,76.650,3.16,2354.35
5,Brebes,15.60,6.420,70.340,8.35,3398.22
6,Cilacap,10.68,7.430,74.685,7.83,3750.29
7,Demak,11.89,8.305,75.975,4.75,2596.38
8,Grobogan,11.43,7.335,75.470,3.23,2837.80
9,Jepara,6.09,8.280,76.465,3.34,2491.75


In [ ]:
W_yes = final.copy().astype(float)

In [ ]:
y = spasial_df['Y']
X = spasial_df[['RLS', 'AHH', 'TPT', 'APD']].copy()

In [ ]:
WX_RLS = W_yes.dot(spasial_df['RLS'])
WX_AHH = W_yes.dot(spasial_df['AHH'])
WX_TPT = W_yes.dot(spasial_df['TPT'])
WX_APD = W_yes.dot(spasial_df['APD'])

In [ ]:
X.loc[:, 'WX_RLS'] = WX_RLS
X.loc[:, 'WX_AHH'] = WX_AHH
X.loc[:, 'WX_TPT'] = WX_TPT
X.loc[:, 'WX_APD'] = WX_APD

In [ ]:
X = sm.add_constant(X)

In [ ]:
model = sm.OLS(y, X).fit()

In [ ]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.555
Model:                            OLS   Adj. R-squared:                  0.418
Method:                 Least Squares   F-statistic:                     4.057
Date:                Thu, 06 Nov 2025   Prob (F-statistic):            0.00304
Time:                        08:11:03   Log-Likelihood:                -75.813
No. Observations:                  35   AIC:                             169.6
Df Residuals:                      26   BIC:                             183.6
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         21.5407     89.861      0.240      0.8

In [ ]:
residuals = model.resid
resid_standardized = (residuals - np.mean(residuals)) / np.std(residuals)

stat, p = kstest(resid_standardized, 'norm')
print(f"KS Statistic = {stat:.3f}, p-value = {p:.3f}")

KS Statistic = 0.103, p-value = 0.818


# **VIF**

In [ ]:
vif_data = pd.DataFrame({
    "Variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif_data)

  Variable           VIF
0    const  47114.374016
1      RLS      3.447517
2      AHH      5.732164
3      TPT      3.790036
4      APD      1.611660
5   WX_RLS      6.783859
6   WX_AHH     13.340033
7   WX_TPT      6.603824
8   WX_APD      1.957847


# **COBA2 OLS**

In [ ]:
y = spasial_df['Y']
X = spasial_df[['RLS', 'AHH', 'TPT', 'APD']].copy()
X = sm.add_constant(X)

In [ ]:
model2 = sm.OLS(y,X).fit()

In [ ]:
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.488
Model:                            OLS   Adj. R-squared:                  0.420
Method:                 Least Squares   F-statistic:                     7.160
Date:                Thu, 06 Nov 2025   Prob (F-statistic):           0.000358
Time:                        08:22:55   Log-Likelihood:                -78.261
No. Observations:                  35   AIC:                             166.5
Df Residuals:                      30   BIC:                             174.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         51.1982     33.741      1.517      0.1

# **REVISI**

Terdapat revisi berdasarkan hasil uji moran yang tidak signifikan, maka dari itu akan dilakukan handling outliers

## **Handling Outliers**

In [ ]:
numeric = spasial_df.select_dtypes(include='number')
Q1 = numeric.quantile(0.25)
Q3 = numeric.quantile(0.75)
IQR = Q3 - Q1
df_no_outliers = spasial_df[~((numeric < (Q1 - 1.5 * IQR)) | (numeric > (Q3 + 1.5 * IQR))).any(axis=1)]
df_no_outliers["Index"] = df_no_outliers.index

/tmp/ipython-input-1297269212.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_outliers["Index"] = df_no_outliers.index


In [ ]:
final_df = df_no_outliers.join(df.set_index('Index'), on='Index', validate='m:1').drop(columns = ["NAMOBJ"])
final_df = final_df.reset_index(drop=True)
final_df.head()

,Kab/Kota,Y,RLS,AHH,TPT,APD,Index,centroid,centroid_lon,centroid_lat
0,Banjarnegara,14.71,6.910,74.875,5.57,2352.02,0,POINT (109.65705 -7.35133),109.657052,-7.351327
1,Batang,8.73,7.175,75.295,5.67,1943.91,2,POINT (109.86159 -7.021),109.861593,-7.021002
2,Blora,11.42,7.330,75.145,3.67,2624.07,3,POINT (111.38744 -7.07444),111.387435,-7.074438
3,Boyolali,9.63,8.215,76.650,3.16,2354.35,4,POINT (110.6525 -7.41652),110.652499,-7.416522
4,Cilacap,10.68,7.430,74.685,7.83,3750.29,6,POINT (108.88984 -7.48893),108.889839,-7.488927


## **Weight Matrix**

In [ ]:
coords = final_df[['centroid_lon', "centroid_lat"]].to_numpy()
n = len(coords)
dist = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist[i, j] = haversine(coords[i,0], coords[i,1], coords[j,0], coords[j,1])

In [ ]:
dist.shape

(29, 29)

In [ ]:
w = Queen.from_dataframe(gdf)

/tmp/ipython-input-38079147.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)


In [ ]:
dict_neighbor = w.neighbors
dict_nbaru = {}
for i in dict_neighbor:
  if i in list(final_df["Index"]):
    dict_nbaru[i] = dict_neighbor[i]

final_dict = {}
for i in dict_nbaru:
  get_value = itemgetter(i)
  val = get_value(dict_nbaru)
  dummy = []
  for j in val:
    if j in list(final_df["Index"]):
      dummy.append(j)
  final_dict[i] = dummy
print(final_dict)

{0: [2, 34, 23, 25, 11], 2: [0, 34, 23, 12], 3: [8, 27, 22], 4: [8, 10, 13, 18, 21, 28, 29, 30], 6: [11], 7: [20, 8, 9, 28], 8: [3, 4, 20, 22, 7, 28, 29], 9: [20, 22, 7], 10: [33, 18, 4, 29, 30], 11: [0, 34, 6, 26], 12: [32, 2, 34, 28], 13: [4, 21, 30], 18: [10, 4, 30], 19: [31], 20: [8, 9, 22, 7], 21: [32, 34, 4, 26, 28, 13], 22: [3, 20, 8, 9, 27], 23: [0, 2, 24, 25], 24: [31, 25, 23], 25: [0, 24, 23], 26: [34, 11, 21], 27: [3, 22], 28: [32, 4, 7, 8, 12, 21], 29: [8, 10, 4], 30: [33, 18, 4, 10, 13], 31: [24, 19], 32: [34, 12, 28, 21], 33: [10, 30], 34: [0, 32, 2, 21, 26, 11, 12]}


In [ ]:
np.isnan(dist).any()

np.False_

In [ ]:
old_indices = list(df_no_outliers["Index"])
new_indices = list(range(29))

# make a mapping dictionary
mapping = dict(zip(old_indices, new_indices))

# rebuild the dictionary with new keys and remapped values
new_neighbors = {
    mapping[k]: [mapping[v] for v in vals if v in mapping]
    for k, vals in final_dict.items()
    if k in mapping
}
new_neighbors

{0: [1, 28, 17, 19, 9],
 1: [0, 28, 17, 10],
 2: [6, 21, 16],
 3: [6, 8, 11, 12, 15, 22, 23, 24],
 4: [9],
 5: [14, 6, 7, 22],
 6: [2, 3, 14, 16, 5, 22, 23],
 7: [14, 16, 5],
 8: [27, 12, 3, 23, 24],
 9: [0, 28, 4, 20],
 10: [26, 1, 28, 22],
 11: [3, 15, 24],
 12: [8, 3, 24],
 13: [25],
 14: [6, 7, 16, 5],
 15: [26, 28, 3, 20, 22, 11],
 16: [2, 14, 6, 7, 21],
 17: [0, 1, 18, 19],
 18: [25, 19, 17],
 19: [0, 18, 17],
 20: [28, 9, 15],
 21: [2, 16],
 22: [26, 3, 5, 6, 10, 15],
 23: [6, 8, 3],
 24: [27, 12, 3, 8, 11],
 25: [18, 13],
 26: [28, 10, 22, 15],
 27: [8, 24],
 28: [0, 26, 1, 15, 20, 9, 10]}

In [ ]:
yesyes = np.zeros((29,29))
yes = new_neighbors
for i in yes:
  for j in yes[i]:
    yesyes[i][j] = dist[i][j]

In [ ]:
def standard(array):
  for i,j in enumerate(array):
    array[i] = j/sum(j)
  return array

In [ ]:
final = standard(yesyes)

# **Uji Moran**

$I = \frac{n}{W} \cdot \frac{\sum_i \sum_j w_{ij} (x_i - \bar{x})(x_j - \bar{x})}{\sum_i (x_i - \bar{x})^2}$

In [ ]:
W_array = final.copy().astype(float)
y = df_no_outliers['Y'].values

neighbors = {i: np.where(W_array[i] > 0)[0].tolist() for i in range(W_array.shape[0])}
weights = {i: W_array[i, W_array[i] > 0].tolist() for i in range(W_array.shape[0])}

w = W(neighbors, weights)
w.transform = "R"

moran = Moran(y, w, permutations=999)

print("Moran’s I:", moran.I)
print("Expected I:", moran.EI)
print("Z-score:", moran.z_sim)
print("P-value:", moran.p_sim)

Moran’s I: 0.13716075653230575
Expected I: -0.03571428571428571
Z-score: 1.2840211857633
P-value: 0.104


In [ ]:
W = final.copy().astype(float)

In [ ]:
y = df_no_outliers['Y']
X = df_no_outliers[['RLS', 'AHH', 'TPT', 'APD']].copy()

In [ ]:
WX_RLS = W.dot(df_no_outliers['RLS'])
WX_AHH = W.dot(df_no_outliers['AHH'])
WX_TPT = W.dot(df_no_outliers['TPT'])
WX_APD = W.dot(df_no_outliers['APD'])

In [ ]:
X.loc[:, 'WX_RLS'] = WX_RLS
X.loc[:, 'WX_AHH'] = WX_AHH
X.loc[:, 'WX_TPT'] = WX_TPT
X.loc[:, 'WX_APD'] = WX_APD

In [ ]:
X = sm.add_constant(X)

In [ ]:
model = sm.OLS(y, X).fit()

In [ ]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.360
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     1.407
Date:                Thu, 06 Nov 2025   Prob (F-statistic):              0.253
Time:                        08:11:09   Log-Likelihood:                -63.732
No. Observations:                  29   AIC:                             145.5
Df Residuals:                      20   BIC:                             157.8
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         81.5093    106.525      0.765      0.4

In [ ]:
residuals = model.resid
resid_standardized = (residuals - np.mean(residuals)) / np.std(residuals)

stat, p = kstest(resid_standardized, 'norm')
print(f"KS Statistic = {stat:.3f}, p-value = {p:.3f}")

KS Statistic = 0.103, p-value = 0.885


# **VIF**

In [ ]:
vif_data = pd.DataFrame({
    "Variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif_data)

  Variable           VIF
0    const  47813.309871
1      RLS      3.195924
2      AHH      5.213996
3      TPT      3.091498
4      APD      2.034126
5   WX_RLS      6.264719
6   WX_AHH     10.518175
7   WX_TPT      5.091233
8   WX_APD      1.342740
